# Gaussian elimination and eigenvalues of a tridiagonal matrix

## a. Gaussian elimination with partial pivoting
Note:
- I want to solve A x = b.
- Gaussian elimination turns A into an upper triangular matrix. Then I can solve it from the last row up (back substitution).
- For column j, I subtract a multiple of row j from every row below it, so the elements below the diagonal become 0. The multiple is m = A_ij / A_jj.
- A_jj is the pivot. If it is 0 (or very small) the division fails. Partial pivoting: before eliminating column j, I swap row j with the row (from j down) that has the largest |A_ij|. Swapping two rows only changes the order of the equations, so the solution stays the same.
- Back substitution, starting from the last row:

$$x_i = \frac{1}{A_{ii}}\left(b_i - \sum_{j=i+1}^{n} A_{ij}\,x_j\right)$$

In [1]:
import numpy as np

In [2]:
# Gaussian elimination with partial pivoting
# A: n x n coefficient matrix, b: right-hand side
# Returns the solution x of A x = b
def gauss_solve(A, b):
    A = np.array(A, dtype=float) #copy, so the original matrix is not changed
    b = np.array(b, dtype=float)
    n = len(b)

    # Elimination: make everything below the diagonal 0, column by column
    for j in range(n - 1):
        # Partial pivoting: find the row (from j down) with the largest |A[i, j]| and swap it with row j
        p = j + np.argmax(np.abs(A[j:, j]))
        A[[j, p]] = A[[p, j]]
        b[[j, p]] = b[[p, j]]

        # Subtract a multiple of row j from every row below, so A[i, j] becomes 0
        for i in range(j + 1, n):
            m = A[i, j] / A[j, j]
            A[i, j:] -= m * A[j, j:]
            b[i] -= m * b[j]

    # Back substitution: start from the last row and go up
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - A[i, i+1:] @ x[i+1:]) / A[i, i]
    return x

I test it on a system where A_11 = 0, so it only works with pivoting:

y + z = 5\
x + z = 4\
x + y = 3

Analytically: adding all three equations gives 2(x + y + z) = 12, so x + y + z = 6. Then x = 6 - 5 = 1, y = 6 - 4 = 2 and z = 6 - 3 = 3.

In [3]:
A = [[0, 1, 1],
     [1, 0, 1],
     [1, 1, 0]]
b = [5, 4, 3]

x = gauss_solve(A, b)
print("numerical: ", x)
print("analytical:", [1, 2, 3])

numerical:  [1. 2. 3.]
analytical: [1, 2, 3]


The numerical result is the same as the analytical one: x = 1, y = 2, z = 3.

## b. Eigenvalues of a symmetric tridiagonal matrix
Note:
- A symmetric tridiagonal matrix has a_1, ..., a_n on the diagonal and b_1, ..., b_(n-1) right above and below it. Everything else is 0.
- The eigenvalues are the roots of the characteristic polynomial p_n(λ) = det(A - λI).
- p_i(λ) is the determinant of the top-left i x i part of A - λI. Expanding this determinant along its last row gives the recursion (Eq. 5.79 in the book):

$$p_i(\lambda) = (a_i - \lambda)\,p_{i-1}(\lambda) - b_{i-1}^2\,p_{i-2}(\lambda)$$

with p_0 = 1 and p_1 = a_1 - λ.
- All eigenvalues lie in [-A_max, A_max], where A_max is the largest sum of |A_ij| over a column (Eq. 5.80 in the book).
- So I scan λ over this interval, look where p_n changes sign, and use bisection on each of these intervals to find the root.
- If two eigenvalues are inside the same step of the scan, p_n does not change sign there and I miss them. So I print how many I found, to check that I have all n.

In [4]:
# Characteristic polynomial p_n(λ) with the recursion (Eq. 5.79)
# a: diagonal (length n), b: off-diagonal (length n-1)
# p_i = (a_i - λ) p_(i-1) - b_(i-1)^2 p_(i-2), with p_0 = 1 and p_1 = a_1 - λ
# Python counts from 0, so a[i] is a_(i+1) and b[i-1] is b_i
# Each p_i only needs the previous two, so I keep them in p_prev and p
def char_poly(a, b, lam):
    p_prev = 1.0       #p_0
    p = a[0] - lam     #p_1
    for i in range(1, len(a)):
        p_prev, p = p, (a[i] - lam)*p - b[i-1]**2 * p_prev
    return p

# Bisection: keep the half where p_n changes sign until the interval is smaller than tol
def bisection(a, b, lo, hi, tol=1e-12):
    while hi - lo > tol:
        mid = (lo + hi)/2
        if char_poly(a, b, lo)*char_poly(a, b, mid) <= 0: #root in the left half
            hi = mid
        else:                                             #root in the right half
            lo = mid
    return (lo + hi)/2

# Find all eigenvalues:
# 1. All eigenvalues are in [-A_max, A_max] (Eq. 5.80)
# 2. Scan λ in small steps and look for sign changes of p_n
# 3. Bisection on every interval with a sign change
def tridiag_eigenvalues(a, b, steps=1000):
    # Largest column sum of |A_ij|: column i has |b_(i-1)|, |a_i| and |b_i|
    col = np.abs(a).copy()
    col[1:] += np.abs(b)     #element above the diagonal
    col[:-1] += np.abs(b)    #element below the diagonal
    A_max = col.max()

    lam = np.linspace(-A_max, A_max, steps)
    p = [char_poly(a, b, l) for l in lam]
    eig = []
    for k in range(steps - 1):
        if p[k]*p[k+1] < 0: #sign change, so there is a root in between
            eig.append(bisection(a, b, lam[k], lam[k+1]))
    return np.array(eig)

I test it on a 5 x 5 matrix with a_i = 2 and b_i = -1, and compare with numpy.

In [5]:
n = 5
a = np.full(n, 2.0)       #diagonal
b = np.full(n - 1, -1.0)  #off-diagonal

eig = tridiag_eigenvalues(a, b)
print("found", len(eig), "of", n, "eigenvalues")
print("recursion + bisection:", eig)

# Full matrix, only to check with numpy
T = np.diag(a) + np.diag(b, 1) + np.diag(b, -1)
print("numpy:                ", np.linalg.eigvalsh(T))

found 5 of 5 eigenvalues
recursion + bisection: [0.26794919 1.         2.         3.         3.73205081]
numpy:                 [0.26794919 1.         2.         3.         3.73205081]


All 5 eigenvalues are found and they are the same as the ones from numpy.